In [1]:
# ================================================================
# PM2.5 Level Prediction using Bidirectional LSTM (All Features)
# ================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

# ---------------- 1) Load Dataset ----------------
train_df = pd.read_csv("train_split.csv")
test_df  = pd.read_csv("test_split.csv")

X_train = train_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_train = train_df["PM2.5_level"].values.astype(np.int64) - 1
X_test  = test_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_test  = test_df["PM2.5_level"].values.astype(np.int64) - 1

num_classes = len(np.unique(y_train))
input_dim = X_train.shape[1]

# ---------------- 2) Scale data ----------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# reshape for LSTM (samples, time_step=1, features)
X_train = X_train.reshape(-1, 1, input_dim)
X_test  = X_test.reshape(-1, 1, input_dim)

# ---------------- 3) Define LSTM model ----------------
class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BiLSTMModel, self).__init__()
        self.lstm1 = nn.LSTM(input_size=input_dim, hidden_size=128, num_layers=1,
                             batch_first=True, bidirectional=True)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = out[:, -1, :]  # last timestep
        out = self.bn1(out)
        out = self.dropout(F.relu(self.fc1(out)))
        out = self.fc2(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ---------------- 4) K-Fold Validation ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"\nFold {fold}...")
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    val_ds   = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False)

    model = BiLSTMModel(input_dim, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    best_val_loss = np.inf
    patience, wait = 8, 0

    for epoch in range(60):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # validation
        model.eval()
        val_loss, val_preds, val_probs, val_true = 0, [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_loss += loss.item()
                probs = F.softmax(out, dim=1).cpu().numpy()
                preds = np.argmax(probs, axis=1)
                val_probs.extend(probs)
                val_preds.extend(preds)
                val_true.extend(yb.cpu().numpy())

        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = model.state_dict().copy()
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_weights)

    acc = accuracy_score(val_true, val_preds)
    prec = precision_score(val_true, val_preds, average='weighted', zero_division=0)
    rec = recall_score(val_true, val_preds, average='weighted', zero_division=0)
    f1 = f1_score(val_true, val_preds, average='weighted', zero_division=0)
    y_bin = label_binarize(val_true, classes=np.arange(num_classes))
    try:
        roc_auc = roc_auc_score(y_bin, np.array(val_probs), average='weighted', multi_class='ovr')
    except:
        roc_auc = np.nan

    val_metrics.append({
        "Fold": fold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

validation_table = pd.DataFrame(val_metrics)
print("\nValidation Results:")
print(validation_table)


Fold 1...

Fold 2...

Fold 3...

Fold 4...

Fold 5...

Validation Results:
   Fold  Accuracy  Precision    Recall  F1-score   ROC-AUC
0     1  0.776640   0.765161  0.776640  0.768200  0.957990
1     2  0.776079   0.764647  0.776079  0.767248  0.958261
2     3  0.774304   0.762506  0.774304  0.765861  0.957766
3     4  0.777759   0.767311  0.777759  0.770108  0.958512
4     5  0.774650   0.763143  0.774650  0.766261  0.957614


In [3]:
# ---------------- 5) Train on full train set ----------------
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

final_model = BiLSTMModel(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(final_model.parameters(), lr=5e-4, weight_decay=1e-4)

final_model.train()
for epoch in tqdm(range(40), desc="Training full model"):
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = final_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

Training full model: 100%|██████████| 40/40 [04:45<00:00,  7.14s/it]


In [4]:
# ---------------- 6) Test Evaluation ----------------
final_model.eval()
with torch.no_grad():
    X_t = torch.tensor(X_test).to(device)
    out = final_model(X_t)
    y_test_proba = F.softmax(out, dim=1).cpu().numpy()
    y_test_pred = np.argmax(y_test_proba, axis=1)

acc_t = accuracy_score(y_test, y_test_pred)
prec_t = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
rec_t = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1_t = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))
try:
    roc_auc_t = roc_auc_score(y_test_bin, y_test_proba, average='weighted', multi_class='ovr')
except:
    roc_auc_t = np.nan

test_table = pd.DataFrame([{
    "Accuracy": acc_t,
    "Precision": prec_t,
    "Recall": rec_t,
    "F1-score": f1_t,
    "ROC-AUC": roc_auc_t
}])

print("\nTest Results:")
print(test_table)


Test Results:
   Accuracy  Precision   Recall  F1-score   ROC-AUC
0   0.77458   0.762971  0.77458  0.765854  0.957701


In [ ]:
# ---------------- 7) Save Results ----------------
validation_table.to_csv("BS_allclass_validation_metrics_lstm.csv", index=False)
test_table.to_csv("BS_allclass_test_metrics_lstm.csv", index=False)

proba_df = pd.DataFrame(y_test_proba, columns=[f"prob_{c}" for c in np.unique(y_train)])
proba_df.insert(0, "true", y_test)
proba_df.to_csv("BS_allclass_proba_test_lstm.csv", index=False)

print("\nSaved: BS_allclass_validation_metrics_lstm.csv, BS_allclass_test_metrics_lstm.csv, BS_allclass_proba_test_lstm.csv")